# Setup

Import library dan dataset dipakai bersama oleh Task 1 dan Task 2.


In [ ]:
# download my recently edited files from GitHub if running this notebook in Google Colab
# !mkdir -p src
# !wget -N -P src https://raw.githubusercontent.com/teranixbq/SequenceModellingNLP/refs/heads/main/src/char_level.py
# !wget -N -P src https://raw.githubusercontent.com/teranixbq/SequenceModellingNLP/refs/heads/main/src/word_level.py

In [ ]:
import re
from collections import Counter
from pathlib import Path

import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, TensorDataset, random_split

from src.word_level import TextCNN, HierarchicalTextCNN
from src.char_level import CharCNN, HierarchicalCharCNN


## Download dan Baca Dataset


In [ ]:
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

csv_path = next(Path(path).glob("*.csv"))
df = pd.read_csv(csv_path)

reviews = df["review"].tolist()
labels = (df["sentiment"] == "positive").astype(int).tolist()


print("Jumlah data:", len(reviews))
print("Contoh label:", labels[0])


# Task 1 - Word-level 1D CNN

## Tokenisasi, Vocabulary, dan Encoding

Review harus diubah menjadi angka: tokenisasi memecah kalimat jadi kata, vocabulary memberi ID untuk setiap kata, encoding mengubah review menjadi deretan ID dengan panjang yang sama.


In [ ]:
def tokenize(text):
    return re.findall(r"[A-Za-z0-9']+", text.lower())

def encode_review(text, word_to_id, max_length):
    token_ids = [word_to_id.get(token, word_to_id["<unk>"]) for token in tokenize(text)]
    token_ids = token_ids[:max_length]
    token_ids += [word_to_id["<pad>"]] * (max_length - len(token_ids))
    return token_ids

In [ ]:
max_vocab_size = 50000
max_length = 200
batch_size = 32
num_epochs = 3

word_counter = Counter()
for review in reviews:
    word_counter.update(tokenize(review))

word_to_id = {"<pad>": 0, "<unk>": 1}

for word, _ in word_counter.most_common(max_vocab_size - 2):
    word_to_id[word] = len(word_to_id)

vocab_size = len(word_to_id)
print("Vocab size:", vocab_size)


## Buat DataLoader

In [ ]:
encoded_reviews = [encode_review(review, word_to_id, max_length) for review in reviews]

batch_x = torch.tensor(encoded_reviews, dtype=torch.long)
batch_y = torch.tensor(labels, dtype=torch.long)

dataset = TensorDataset(batch_x, batch_y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

print("Shape X:", batch_x.shape)
print("Shape y:", batch_y.shape)
print("Train data:", len(train_dataset))
print("Validation data:", len(val_dataset))


## Compare Pooling Performance

Bagian akhir ini langsung membandingkan `max`, `avg`, `none` atau without pooling, dan optional `adaptive`.


In [ ]:
from time import perf_counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


def train_and_evaluate(pooling_type, kernel_sizes = [3, 4]):
    model = TextCNN(
        vocab_size=vocab_size,
        embed_dim=300,
        num_filters=100,
        kernel_sizes=kernel_sizes,
        num_classes=2,
        pooling_type=pooling_type,
        sequence_length=max_length,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print(f"Start training pooling: {pooling_type}, Kernel sizes: {kernel_sizes}", flush=True)

    if device == "cuda":
        torch.cuda.synchronize()

    start_time = perf_counter()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch_idx, (batch_x, batch_y) in enumerate(train_dataloader, start=1):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 50 == 0 or batch_idx == len(train_dataloader):
                print(
                    f"Pooling: {pooling_type}, Epoch {epoch + 1}/{num_epochs}, "
                    f"Batch {batch_idx}/{len(train_dataloader)}, "
                    f"Loss: {total_loss / batch_idx:.4f}",
                    flush=True,
                )

    if device == "cuda":
        torch.cuda.synchronize()

    training_time = perf_counter() - start_time

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch_x, batch_y in val_dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = probabilities.argmax(dim=1)

            y_true += batch_y.cpu().tolist()
            y_pred += predictions.cpu().tolist()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)


    return {
        "model_type": "word_cnn",
        "pooling_type": pooling_type,
        "kernel_sizes": kernel_sizes,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "training_time": training_time,
    }


In [ ]:
results = []

for pooling_type in ["max", "avg", "none", "adaptive"]:
    record = train_and_evaluate(pooling_type)
    results.append(record)

    print(
        f"Model: {record['model_type']}, "
        f"Pooling: {record['pooling_type']}, "
        f"Kernel Sizes: {record['kernel_sizes']}, "
        f"Accuracy: {record['accuracy']:.4f}, "
        f"Precision: {record['precision']:.4f}, "
        f"Recall: {record['recall']:.4f}, "
        f"F1: {record['f1_score']:.4f}, "
        f"Training Time: {record['training_time']:.2f}s",
        flush=True,
    )

### Compare World - CNN with 4 kernel sizes

Meenggunaakn kernel size lain untuk membandingkan dengan compare sebelumnya apakah ada perbedaan atau tidak

In [ ]:
for pooling_type in ["max", "avg", "none", "adaptive"]:
    record = train_and_evaluate(pooling_type, kernel_sizes=[3, 4, 5, 6])
    results.append(record)

    print(
        f"Model: {record['model_type']}, "
        f"Pooling: {record['pooling_type']}, "
        f"Kernel Sizes: {record['kernel_sizes']}, "
        f"Accuracy: {record['accuracy']:.4f}, "
        f"Precision: {record['precision']:.4f}, "
        f"Recall: {record['recall']:.4f}, "
        f"F1: {record['f1_score']:.4f}, "
        f"Training Time: {record['training_time']:.2f}s",
        flush=True,
    )

### Hierarchical Word CNN

Bagian ini memakai beberapa Conv1D word-level yang disusun bertingkat.

### Train and Evaluate Hierarchical Word CNN

Fungsi ini mengevaluasi `HierarchicalTextCNN` dengan metrik yang sama seperti eksperimen pooling utama.


In [ ]:
def train_and_evaluate_hierarchical(kernel_sizes=[3, 5, 7]):
    model = HierarchicalTextCNN(
        vocab_size=vocab_size,
        embed_dim=300,
        num_filters=100,
        kernel_sizes=kernel_sizes,
        num_classes=2,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print(f"Start training Hierarchical CNN with max pooling and kernel sizes {kernel_sizes}", flush=True)

    if device == "cuda":
        torch.cuda.synchronize()

    start_time = perf_counter()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch_idx, (batch_x, batch_y) in enumerate(train_dataloader, start=1):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 50 == 0 or batch_idx == len(train_dataloader):
                print(
                    f"Hierarchical CNN, Epoch {epoch + 1}/{num_epochs}, "
                    f"Batch {batch_idx}/{len(train_dataloader)}, "
                    f"Loss: {total_loss / batch_idx:.4f}",
                    flush=True,
                )

    if device == "cuda":
        torch.cuda.synchronize()

    training_time = perf_counter() - start_time

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch_x, batch_y in val_dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = probabilities.argmax(dim=1)

            y_true += batch_y.cpu().tolist()
            y_pred += predictions.cpu().tolist()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    return {
        "model_type": "hierarchical_word_cnn",
        "pooling_type": "max",
        "kernel_sizes": kernel_sizes,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "training_time": training_time,
    }

    

### Run Hierarchical Word CNN

In [ ]:
if "results" not in globals():
    results = []

hierarchical_record = train_and_evaluate_hierarchical()
results = [
    record
    for record in results
    if record.get("model_type") != "hierarchical_word_cnn"
]
results.append(hierarchical_record)

### Top 5 Ranking di word CNN

`task1_results` diambil dari `results`, lalu diurutkan berdasarkan accuracy.


In [ ]:
task1_results = sorted(
    results if "results" in globals() else [],
    key=lambda record: record["accuracy"],
    reverse=True,
)
task1_top5_results = task1_results[:5]
result_1 = task1_top5_results

pd.DataFrame(result_1)


## Task 2 - Character CNN

- Convert text into character sequences (no word tokenization) 
- Build character embeddings 
- Apply 1D CNN over character sequences to create word-level representations 
- Compare the performance of Character CNN vs Word CNN on the same dataset, 
- especially on sentences containing slang, typos, or rare words.

### Character Configuration

`char_max_length` membatasi panjang review dalam satuan karakter agar semua input punya shape yang sama.


In [ ]:
char_max_length = 500


### Character Vocabulary

Task 2 tidak memakai word tokenization. Setiap review dibaca sebagai urutan karakter, lalu setiap karakter diberi ID.


In [ ]:
char_to_id = {"<pad>": 0, "<unk>": 1}

for review in reviews:
    for char in review.lower():
        if char not in char_to_id:
            char_to_id[char] = len(char_to_id)

char_vocab_size = len(char_to_id)
print("Char vocab size:", char_vocab_size)


### Character Encoding

Review dipotong sampai `char_max_length`, lalu dipadding dengan ID `<pad>` jika lebih pendek.


In [ ]:
def encode_review_chars(text, char_to_id, max_length):
    char_ids = [
        char_to_id.get(char, char_to_id["<unk>"])
        for char in text.lower()[:max_length]
    ]
    char_ids += [char_to_id["<pad>"]] * (max_length - len(char_ids))
    return char_ids


encoded_char_reviews = [
    encode_review_chars(review, char_to_id, char_max_length)
    for review in reviews
]

char_batch_x = torch.tensor(encoded_char_reviews, dtype=torch.long)
char_batch_y = torch.tensor(labels, dtype=torch.long)

print("Char X:", char_batch_x.shape)
print("Char y:", char_batch_y.shape)


### Character DataLoader

Character CNN memakai split train/validation yang sama dengan Word CNN agar perbandingannya fair.


In [ ]:
char_dataset = TensorDataset(char_batch_x, char_batch_y)
char_train_dataset = Subset(char_dataset, train_dataset.indices)
char_val_dataset = Subset(char_dataset, val_dataset.indices)

char_train_dataloader = DataLoader(
    char_train_dataset,
    batch_size=batch_size,
    shuffle=True,
)
char_val_dataloader = DataLoader(
    char_val_dataset,
    batch_size=batch_size,
)

print("Char train data:", len(char_train_dataset))
print("Char validation data:", len(char_val_dataset))


### Character CNN Training Setup

Bagian ini menyiapkan device dan metrik evaluasi untuk Character CNN.


In [ ]:
from time import perf_counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


### Train and Evaluate Character CNN

Fungsi ini bisa menjalankan `CharCNN` biasa atau `HierarchicalCharCNN` untuk bonus optional.


In [ ]:
def train_and_evaluate_char(pooling_type="max", hierarchical=False):
    model_class = HierarchicalCharCNN if hierarchical else CharCNN
    model_type = "hierarchical_char_cnn" if hierarchical else "char_cnn"
    kernel_sizes = [3, 5, 7] if hierarchical else [3, 5]

    if hierarchical:
        pooling_type = "max"

    if hierarchical:
        model = model_class(
            char_vocab_size=char_vocab_size,
            char_embed_dim=64,
            num_filters=100,
            kernel_sizes=kernel_sizes,
            num_classes=2,
        ).to(device)
    else:
        model = model_class(
            char_vocab_size=char_vocab_size,
            char_embed_dim=64,
            num_filters=100,
            kernel_sizes=kernel_sizes,
            num_classes=2,
            pooling_type=pooling_type,
            sequence_length=char_max_length,
        ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print(
        f"Start training {model_type} with {pooling_type} pooling "
        f"and kernel sizes {kernel_sizes}",
        flush=True,
    )

    if device == "cuda":
        torch.cuda.synchronize()

    start_time = perf_counter()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch_idx, (batch_x, batch_y) in enumerate(char_train_dataloader, start=1):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 50 == 0 or batch_idx == len(char_train_dataloader):
                print(
                    f"Model: {model_type}, Epoch {epoch + 1}/{num_epochs}, "
                    f"Batch {batch_idx}/{len(char_train_dataloader)}, "
                    f"Loss: {total_loss / batch_idx:.4f}",
                    flush=True,
                )

    if device == "cuda":
        torch.cuda.synchronize()

    training_time = perf_counter() - start_time

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch_x, batch_y in char_val_dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = probabilities.argmax(dim=1)

            y_true += batch_y.cpu().tolist()
            y_pred += predictions.cpu().tolist()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return {
        "model_type": model_type,
        "pooling_type": pooling_type,
        "kernel_sizes": kernel_sizes,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "training_time": training_time,
    }


### Run Character CNN

Jalankan Character CNN biasa dengan beberapa pooling type, seperti eksperimen pooling pada Task 1.


In [ ]:
char_results = []

for pooling_type in ["max", "avg", "none", "adaptive"]:
    record = train_and_evaluate_char(pooling_type=pooling_type)
    char_results.append(record)

    print(
        f"Model: {record['model_type']}, "
        f"Pooling: {record['pooling_type']}, "
        f"Kernel Sizes: {record['kernel_sizes']}, "
        f"Accuracy: {record['accuracy']:.4f}, "
        f"Precision: {record['precision']:.4f}, "
        f"Recall: {record['recall']:.4f}, "
        f"F1: {record['f1_score']:.4f}, "
        f"Training Time: {record['training_time']:.2f}s",
        flush=True,
    )

pd.DataFrame(char_results)


### Hierarchical Character CNN

Bagian ini memakai `HierarchicalCharCNN`, yaitu beberapa Conv1D karakter yang disusun bertingkat. Jalankan cell ini jika ingin mengambil bonus/eksperimen hierarchical untuk Task 2.


In [ ]:
hierarchical_char_record = train_and_evaluate_char(hierarchical=True)

pd.DataFrame([hierarchical_char_record])


## Compare Word CNN vs Character CNN

Bagian ini diisi nanti setelah Task 2 selesai. Perbandingannya memakai dataset yang sama, terutama untuk teks slang, typo, atau rare words.


### Task 2 Top 5 Ranking

`task2_results` menggabungkan model Word CNN terbaik dari Task 1, Character CNN, dan Hierarchical Character CNN jika sudah dijalankan.


In [ ]:
task2_results = []

if "results" in globals() and results:
    best_word_record = max(results, key=lambda record: record["accuracy"])
    task2_results.append({
        "model_type": best_word_record.get("model_type", "word_cnn"),
        "pooling_type": best_word_record["pooling_type"],
        "kernel_sizes": best_word_record.get("kernel_sizes"),
        "accuracy": best_word_record["accuracy"],
        "precision": best_word_record["precision"],
        "recall": best_word_record["recall"],
        "f1_score": best_word_record["f1_score"],
        "training_time": best_word_record["training_time"],
    })

if "char_results" in globals() and char_results:
    task2_results.extend(char_results)

if "hierarchical_char_record" in globals():
    task2_results.append(hierarchical_char_record)

task2_results = sorted(
    task2_results,
    key=lambda record: record["accuracy"],
    reverse=True,
)
task2_top5_results = task2_results[:5]
result_2 = task2_top5_results

pd.DataFrame(result_2)
